In [0]:
!pip install lightgbm
import mlflow
from mlflow import MlflowClient

print("MLflow version:", mlflow.__version__)

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
MLflow version: 3.8.1


In [0]:
CATALOG = "workspace"
SCHEMA = "default"

MODEL_NAME = f"{CATALOG}.{SCHEMA}.demand_forecasting"

EXPERIMENT_NAME = "/Shared/demand-forecasting"

mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()

print("Experiment :", EXPERIMENT_NAME)
print("Model      :", MODEL_NAME)
print("Registry   : databricks-uc")

Experiment : /Shared/demand-forecasting
Model      : workspace.default.demand_forecasting
Registry   : databricks-uc


In [0]:
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise ValueError(
        f"Experiment '{EXPERIMENT_NAME}' was not found."
    )

EXPERIMENT_ID = experiment.experiment_id

print("Experiment ID:", EXPERIMENT_ID)

Experiment ID: 2993782678277182


In [0]:
runs = client.search_runs(
    experiment_ids=[EXPERIMENT_ID],
    filter_string="status = 'FINISHED'",
    order_by=["start_time DESC"],
    max_results=20
)

if not runs:
    raise ValueError(
        "No successful MLflow runs found in the experiment."
    )

print(f"Found {len(runs)} completed runs.")

Found 15 completed runs.


In [0]:
for run in runs:
    print(
        f"Run ID: {run.info.run_id} | "
        f"Run Name: {run.data.tags.get('mlflow.runName')} | "
        f"Start Time: {run.info.start_time}"
    )

Run ID: 8ed160026e9043538a658c8016a1067f | Run Name: lightgbm_demand_forecasting | Start Time: 1788433352321
Run ID: 3ee188442f354bd4a25af5eceabe1730 | Run Name: lightgbm_demand_forecasting | Start Time: 1788433319428
Run ID: 301af51a4a0b41e5ad60340e156999ae | Run Name: lightgbm_demand_forecasting | Start Time: 1788433173007
Run ID: e0708eff9fb04197883e8c497a8f90e7 | Run Name: lightgbm_demand_forecasting | Start Time: 1788359043200
Run ID: 43eb761a85aa4bff84c3d393c5a82ff5 | Run Name: lightgbm_demand_forecasting | Start Time: 1788358996755
Run ID: 1c0a052144fb4775ab3035398ee689c6 | Run Name: lightgbm_demand_forecasting | Start Time: 1788357237928
Run ID: 8c4dfd015d4243c99367f06e161cae4c | Run Name: lightgbm_demand_forecasting | Start Time: 1788356483859
Run ID: 34e6238c05584a478621b83fa35ee65e | Run Name: lightgbm_with_signature | Start Time: 1788256505990
Run ID: 3d77970b3d5047278c34abe6487410bc | Run Name: lightgbm_baseline | Start Time: 1788256236718
Run ID: 02bff04a25584f84923102458

In [0]:
latest_run = runs[0]

RUN_ID = latest_run.info.run_id

print("Selected Run ID:", RUN_ID)
print("Run Name:", latest_run.data.tags.get("mlflow.runName"))

Selected Run ID: 8ed160026e9043538a658c8016a1067f
Run Name: lightgbm_demand_forecasting


In [0]:
print("Metrics from selected run:")

for key, value in latest_run.data.metrics.items():
    print(f"{key}: {value}")

Metrics from selected run:
baseline_mape: 53.25413025015813
baseline_r2: -0.33896715469405336
baseline_rmse: 51.71874183389165
mae_improvement_pct: 72.14803653926542
mape_improvement_pct: 66.97501839344069
baseline_mae: 39.93761467889908
validation_rmse: 16.849697084376338
validation_r2: 0.8793130435926209
validation_mape: 15.188812339497115
validation_mae: 11.92655913822473
test_rmse: 15.254303021916487
test_r2: 0.8835180158362557
test_mape: 17.587166719847858
test_mae: 11.123409847455939
rmse_improvement_pct: 70.50527046673004


In [0]:
MODEL_URI = f"runs:/{RUN_ID}/model"

print("Model URI:", MODEL_URI)

Model URI: runs:/8ed160026e9043538a658c8016a1067f/model


In [0]:
loaded_model = mlflow.lightgbm.load_model(MODEL_URI)

print("Model loaded successfully.")
print("Model type:", type(loaded_model))

Model loaded successfully.
Model type: <class 'lightgbm.sklearn.LGBMRegressor'>


In [0]:
registered_model = mlflow.register_model(model_uri=MODEL_URI, name=MODEL_NAME)

print("Model registered successfully.")
print("Model name   :", registered_model.name)
print("Model version:", registered_model.version)

Registered model 'workspace.default.demand_forecasting' already exists. Creating a new version of this model...
2026/09/03 11:15:58 WARNING mlflow.tracking._model_registry.fluent: Run with id 8ed160026e9043538a658c8016a1067f has no artifacts at artifact path 'model', registering model based on models:/m-7f54f2a50e594f8880b67482dac20b0d instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '2' of model 'workspace.default.demand_forecasting': https://dbc-1497021e-8b51.cloud.databricks.com/explore/data/models/workspace/default/demand_forecasting/version/2?o=7474655755433327


Model registered successfully.
Model name   : workspace.default.demand_forecasting
Model version: 2


In [0]:
import time

model_version = registered_model.version

for _ in range(30):
    mv = client.get_model_version(
        name=MODEL_NAME,
        version=model_version
    )

    print("Status:", mv.status)

    if mv.status == "READY":
        print("Model version is READY.")
        break

    time.sleep(2)
else:
    print("Model version is still being processed.")

Status: READY
Model version is READY.


In [0]:
client.set_model_version_tag(
    name=MODEL_NAME,
    version=model_version,
    key="model_type",
    value="LightGBM"
)

client.set_model_version_tag(
    name=MODEL_NAME,
    version=model_version,
    key="problem_type",
    value="demand_forecasting"
)

client.set_model_version_tag(
    name=MODEL_NAME,
    version=model_version,
    key="framework",
    value="LightGBM"
)

print("Model version tags added.")

Model version tags added.


In [0]:
client.update_model_version(
    name=MODEL_NAME,
    version=model_version,
    description=(
        "LightGBM demand forecasting model trained using "
        "historical demand lags, rolling statistics, "
        "calendar features, business features, and categorical features."
    )
)

print("Model description updated.")

Model description updated.


In [0]:
model_info = client.get_model_version(
    name=MODEL_NAME,
    version=model_version
)

print("====================================")
print("Registered Model")
print("====================================")
print("Name       :", model_info.name)
print("Version    :", model_info.version)
print("Status     :", model_info.status)
print("Run ID     :", model_info.run_id)
print("Source     :", model_info.source)
print("Description:", model_info.description)
print("====================================")

Registered Model
Name       : workspace.default.demand_forecasting
Version    : 2
Status     : READY
Run ID     : 8ed160026e9043538a658c8016a1067f
Source     : models:/m-7f54f2a50e594f8880b67482dac20b0d
Description: LightGBM demand forecasting model trained using historical demand lags, rolling statistics, calendar features, business features, and categorical features.


In [0]:
versions = client.search_model_versions(
    filter_string=f"name = '{MODEL_NAME}'"
)

print(f"Registered versions for {MODEL_NAME}:\n")

for version in versions:
    print(
        f"Version: {version.version} | "
        f"Status: {version.status} | "
        f"Run ID: {version.run_id}"
    )

Registered versions for workspace.default.demand_forecasting:

Version: 2 | Status: READY | Run ID: 8ed160026e9043538a658c8016a1067f
Version: 1 | Status: READY | Run ID: 34e6238c05584a478621b83fa35ee65e
